# Practical approach to the diet problem using machines

First we want a list of tings to eat but we don't want to write down them like primitives, we want to fill it with some automatism such as an input procedure which accepts strings and depending on the strings does some operations.

In [1]:
tte = []
inmode = True
while inmode == True:
    command = input('q to quit, c to clear,d to deleete, a to add ')
    if command == 'c': tte=[]
    if command == 'a':
        while command == 'a':
            edible=input('commands to command or insert thing to eat ')
            if edible == 'commands': command=0
            else : tte.append(edible)
    elif command == 'd':
        while command == 'd':
            todel = input('commands to command or insert element to delete ')
            if todel == 'commands': command=0
            else:
                try: tte.remove(todel)
                except: print('not in list')
    else: inmode = False

q to quit, c to clear,d to deleete, a to add q


We would like to interact with this list with a function that records the list in alphabetic order and we'd like to load it from a path every time the function is called. Possibly befoere recoding the new thing to eat it may be good to make the function look for similar elements in the list.

In [2]:
def look_for(path,default=[]):
    try:
        with open(path) as f:
            obj = f.read()
            f.close()
    except:
        print('nothing found at',path)
        obj=default
        return default
    return eval(obj)

def store_data(path,data):
    with open(path, 'w') as g:
        g.write(str(data))
        g.close()
    return

def mode_tte(tte):
    inmode = True
    while inmode == True:
        command = input('q to quit, c to clear,d to deleete, a to add ')
        if command == 'c': tte=[]
        if command == 'a':
            while command == 'a':
                edible=input('commands to command or insert thing to eat ')
                if edible == 'commands': command=0
                else : tte.append(edible)
        elif command == 'd':
            while command == 'd':
                todel = input('commands to command or insert element to delete ')
                if todel == 'commands': command=0
                else:
                    try: tte.remove(todel)
                    except: print('not in list')
        else: inmode = False
    tte.sort()
    store_data('tingstoeat.txt',tte)
    return tte

In [3]:
tte=look_for('tingstoeat.txt')

nothing found at tingstoeat.txt


Other than the list we want the foods in it to be related with their nutritional vlues but again we don't want to do this like primitives so we crete a dictionary which does it and a function that fills it.

In [4]:
def foods_and_values(tte,value):
    values = look_for('nvalues.txt', dict())
    for i in tte:
        if ((i in values) is False):
            values[i] = dict()
            print('say how much '+value+' in '+i)
            val = input('or type skip if you do not know ')
            if val=='skip':
                print('ok nevermind :) ')
            else:
                values[i][value] = eval(val)
        elif ((value in values[i]) is False):
            print('say how much '+value+' in '+i)
            val = input('or type skip if you do not know ')
            if val=='skip':
                print('ok nevermind :) ')
            else:
                values[i][value] = eval(val)
    store_data('nvalues.txt',values)
    return values

In [5]:
values = foods_and_values(tte,'cal/hg')

nothing found at nvalues.txt


In order to get a diet we need to know what we need to assume energy from it, so  first we compute how much energy we consume in a day or longer periodes. In order to do do this we measure our mass and calories eaten at different times and make a linear fit of our data in order to get the the mass per time variation.

In [6]:
from datetime import datetime as d

w_today = [d.now(),61.5]

In [7]:
'''this is an sxample of how to compute calories: generally multiplying the mass of 
the food times the calories per unit mass and sometimes just adding some known value of some 
ready made dish'''
calories = 0
calories += 9*50
calories += 350*2.05
calories += 200
calories += 150*3.75
calories += 214

print(calories)

2144.0


Many people are used to consume alcool and since it contain caloroies it may be useful a function that computes the calories of a drink based on its volume and its alcool content.

In [8]:
def from_alch_to_hg(vol,alch):
    alch_dens = 0.789
    total_vol = vol*alch/100
    kg = total_vol*alch_dens
    return kg*10
print(from_alch_to_hg(0.33,8.4))

0.2187108


Sometime we may want to compute the calories of our diet multiplying the mass of what we eat times their calories density so we may usee a function as the following.

Some others we may want to simply add some portions of food with known calories and add it to the rest, so first we make a list of this type of eaten and than a dictionary with the nutritional values of the elements int he list.

In [9]:
portions=look_for('portions.txt')

nothing found at portions.txt


In [10]:
def add_to_eaten(eaten=[]):
    #basically this takes a list of food and quantities and add to if the food and quantity you input
    food = input('name of food ')
    quantity = eval(input('eaten quantity '))
    eaten.append([food,quantity])
    return eaten

def comp_cal_eaten(tte,portions,nvalues,eaten):
    
    calories = 0
    print('input food ')
    while True: 
        eaten = add_to_eaten(eaten)#check this
        on = input('more? (y/n) ')
        if on in ['yes','y','s','sì','si']:
            print('input food ')
        else: break
    for i in eaten:
        if i[0] in tte:
            calories += nvalues[i[0]]['cal/hg']*i[1]
        elif i[0] in portions:
            calories += nvalues[i[0]]['cal']*i[1]
        else:
            print('do not know ',i[0])
            case = input('type a to add to list j to just this once ')
            if case=='a':
                thelist = input('type t to add to things to eat or p if it is a portion')
                cal_of_i = eval(input('type its calories '))
                if thelist=='t':
                    tte.append(i[0])
                    tte.sort()
                    with open('tingstoeat.txt', 'w') as g:
                        g.write(str(tte))
                        g.close()
                    nvalues[i[0]] = {'cal/hg':cal_of_i}
                elif thelist=='p':
                    portions.append(i[0])
                    portions.sort()
                    with open('portions.txt', 'w') as g:
                        g.write(str(portions))
                        g.close()
                        nvalues[i[0]] = {'cal':cal_of_i}
                else:
                    print('did not get it sorry :( ')
                with open('nvalues.txt', 'w') as g:
                    g.write(str(nvalues))
                    g.close()
            elif case=='j':
                cal_of_i = eval(input('type its calories '))
            else:
                print('did not get it sorry :( ')
                cal_of_i = 0
            calories += cal_of_i*i[1]
    return calories,eaten

In [11]:
#calories,eaten = comp_cal_eaten(tte,portions,values,[])

In [12]:
print(calories)

2144.0


Some food may be found in different forms so it may be useful to set them in classes, than calling a class of foods a program should choose which element of the class has to be considered.

In [13]:
f_classes = look_for('foodclasse.txt', dict())

nothing found at foodclasse.txt


In [14]:
def add_to_f_class(f_class,item):
    f_classes = look_for('foodclasse.txt', dict())
    f_classes[f_class] = item
    store_data('foodclasses.txt',f_classes)
    return

## Graphical interface
Ok it's time to go graphical!

In [15]:
from tkinter import*

root = Tk()
root.title('PATTDPUM')
root.geometry('500x300')

the_lable = Label(root, text='name of food ', font=('Helvetica', 15))
the_lable.grid(row = 0, column = 0)

the_entry = Entry(root, font=('Helvetica', 20))
the_entry.grid(row=1, column=0)

options = Listbox(root, width = 40)
options.grid(row=2, column=0)

def check(e):
    typed = the_entry.get()
    if typed == '':
        options.delete(0,END)
        for i in the_list:
            options.insert(END,i)
    else:
        options.delete(0,END)
        for i in the_list:
            if typed.lower() in i.lower(): options.insert(END,i)
    return

def fillout(e):
    the_entry.delete(0,END)
    the_entry.insert(0,options.get(ACTIVE))
    return

the_list=[]

the_entry.bind('<KeyRelease>', check)
options.bind('<<ListboxSelect>>', fillout)

input_list = Listbox(root, width = 20)
input_list.grid(row=2, column=1)


def add_to_lb():
    #basically this takes a list of food and quantities and add to if the food and quantity you input
    food = the_entry.get()
    input_list.insert(END,food)
    return


the_button = Button(root, text='add', command = add_to_lb)
the_button.grid(row=1, column=1)

for i in the_list:
    options.insert(END,i)

root.mainloop()

We define now some functions that can be used with the grafical interface.

In [16]:
def look_for(path,default=[]):
    try:
        with open(path) as f:
            obj = f.read()
            f.close()
    except:
        print('nothing found at',path)
        obj=default
        return obj
    return eval(obj)

def store_data(path,data):
    with open(path, 'w') as g:
        g.write(str(data))
        g.close()
    return

root = Tk()
root.title('PATTDPUM')
root.geometry('900x300')

food_lable = Label(root, text='name of food ', font=('Helvetica', 15))
food_lable.grid(row = 0, column = 0)

food_entry = Entry(root, font=('Helvetica', 14))
food_entry.grid(row=1, column=0)

listed = Listbox(root, width = 40)
listed.grid(row=2, column=0)


In [17]:
from datetime import datetime as d
def date_to_tuple(date):
    t = (date.year,date.month,date.day,date.hour,date.minute,date.second)
    return t
def tuple_to_date(t):
    date = d(t[0],t[1],t[2],t[3],t[4],t[5])
    return date

print(date_to_tuple(d.now()),(tuple_to_date(date_to_tuple(d.now()))-d(1,1,1,0,0)).total_seconds()/3600)

(2026, 4, 16, 14, 23, 20) 17753318.388888888


We then use them in a simple exaple of program that allows the user to record some parameters and shows the data in a diagram.

In [18]:
from tkinter import*
from datetime import datetime as d
import numpy as np
from scipy import stats
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

today = d.now()
root = Tk()
root.title('PATTDPUM')
root.geometry('1050x530')



def state():
    def get_w():
        dates = []
        w = []
        wrec = look_for('wrecord.txt')
        for i in wrec:
            date=d(i[0][0],i[0][1],i[0][2],i[0][3]) -d(wrec[0][0][0],wrec[0][0][1],wrec[0][0][2],wrec[0][0][3])
            dates.append(date.total_seconds()/(3600*24))
            w.append(i[1])
        add_plot(np.array(dates),np.array(w))
        #print(stats.linregress(np.array(dates),np.array(w)))
        return np.array(dates),np.array(w)
    
    w_lable = Label(root, text='weight today', font=('Helvetica', 15))
    w_lable.grid(row = 0, column = 0)

    w_entry = Entry(root, font=('Helvetica', 20))
    w_entry.grid(row=1, column=0)

    def record_weight():
        w_today = [date_to_tuple(d.now()),eval(w_entry.get())]
        
        weights = look_for('wrecord.txt')

        weights.append(w_today)

        with open('wrecord.txt', 'w') as g:
            g.write(str(weights))
            g.close()
        return
    
    def add_plot(x, y):
        fig = Figure(figsize = (4, 4))
        splot = fig.add_subplot(111)
        splot.plot(y)
        canvas = FigureCanvasTkAgg(fig, master = root)  
        canvas.draw()
        canvas.get_tk_widget().grid(column=2)
        return

    w_button = Button(root, text = 'record weight today', command = record_weight)
    w_button.grid(row=2,column=0)

    nrg_lable = Label(root, text='eaten calories today', font=('Helvetica', 15))
    nrg_lable.grid(row = 0, column = 1)

    nrg_entry = Entry(root, font=('Helvetica', 20))
    nrg_entry.grid(row=1, column=1)

    def record_nrg():
        nrg = look_for('nrgrecord.txt')

        nrg.append([date_to_tuple(d.now()),eval(nrg_entry.get())])

        with open('nrgrecord.txt', 'w') as g:
            g.write(str(nrg))
            g.close()

    nrg_button = Button(root, text = 'record calories today', command = record_nrg)
    nrg_button.grid(row=2,column=1)
    
    comp_button = Button(root, text = 'compute calories')
    comp_button.grid()
    get_w()
    
state()

root.mainloop()


nothing found at wrecord.txt


In [19]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import tkinter as tk

# Funzione per passare alla seconda scheda
def avanti():
  # Nascondere la prima scheda
  scheda1.pack_forget()

  # Mostrare la seconda scheda
  scheda2.pack()

# Funzione per tornare alla prima scheda
def indietro():
  # Nascondere la seconda scheda
  scheda2.pack_forget()

  # Mostrare la prima scheda
  scheda1.pack()

# Creazione della finestra principale
root = tk.Tk()

# Creazione delle due schede
scheda1 = tk.Frame(root)
scheda2 = tk.Frame(root)

# Creazione del grafico nella prima scheda
grafico = tk.Canvas(scheda1, width=400, height=300)
grafico.pack()

# Creazione del bottone nella prima scheda
bottone_avanti = tk.Button(scheda1, text="Avanti", command=avanti)
bottone_avanti.pack()


# Creazione della lista nella seconda scheda
lista = tk.Listbox(scheda2)
lista.pack()

# Creazione del bottone nella seconda scheda
bottone_indietro = tk.Button(scheda2, text="Indietro", command=indietro)
bottone_indietro.pack()

# Visualizzazione della prima scheda all'avvio
scheda1.pack()

# Avvio del mainloop
root.mainloop()

In [15]:
from tkinter import*
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from datetime import datetime as d
import numpy as np
from scipy import stats

# Funzione per la prima scheda
def scheda_1():
    # Creazione del grafico
    # ...
    def record_weight():
        '''funzione che registra su un file la data e il peso input'''
        w_today = [date_to_tuple(d.now()),eval(w_entry.get())]

        weights = look_for('wrecord.txt')

        weights.append(w_today)

        with open('wrecord.txt', 'w') as g:
            g.write(str(weights))
            g.close()
        return
    
    def record_nrg():
        '''funzione che registra su un file la data e le calorie mangiate input'''
        nrg = look_for('nrgrecord.txt')

        nrg.append([date_to_tuple(d.now()),eval(nrg_entry.get())])

        with open('nrgrecord.txt', 'w') as g:
            g.write(str(nrg))
            g.close()
        return

    def get_w():
        dates = []
        w = []
        wrec = look_for('wrecord.txt')
        for i in wrec:
            date=d(i[0][0],i[0][1],i[0][2],i[0][3]) -d(wrec[0][0][0],wrec[0][0][1],wrec[0][0][2],wrec[0][0][3])
            dates.append(date.total_seconds()/(3600*24))
            w.append(i[1])
        add_plot(np.array(dates),np.array(w))
        return np.array(dates),np.array(w)
    
    def add_plot(x, y):
        fig = plt.Figure(figsize = (4, 4))
        splot = fig.add_subplot(111)
        splot.plot(y)
        canvas = FigureCanvasTkAgg(fig, master = root)  
        canvas.draw()
        canvas.get_tk_widget().grid(column=2)
        return
    
    for widg in root.winfo_children():
        widg.grid_forget()
    
    w_button = Button(root, text = 'record weight today', command = record_weight)
    w_button.grid(row=2,column=0)

    nrg_lable = Label(root, text='eaten calories today', font=('Helvetica', 15))
    nrg_lable.grid(row = 0, column = 1)

    nrg_entry = Entry(root, font=('Helvetica', 20))
    nrg_entry.grid(row=1, column=1)
    nrg_entry.insert(0,calories)
    
    w_lable = Label(root, text='weight today', font=('Helvetica', 15))
    w_lable.grid(row = 0, column = 0)

    w_entry = Entry(root, font=('Helvetica', 20))
    w_entry.grid(row=1, column=0)
    # Creazione del bottone
    button_1 = Button(root, text="Vai alla scheda 2", command=scheda_2)
    button_1.grid(row=3, column=3)
    
    nrg_button = Button(root, text = 'record calories today', command = record_nrg)
    nrg_button.grid(row=2,column=1)
    
    eaten_lists[-1][1] = eaten_list
    store_data('eaten_lists.txt',eaten_lists)
    
    gra = get_w()
    if 0<len(gra[0]):
        s = stats.linregress(gra[0],gra[1])
        s_label = Label(root, text=
                    str(" A = %.2f" % s[1])+'('+str("%.2f" % s.intercept_stderr)+') B = ' + 
                    str(" B = %.3f" % s[0])+'('+str("%.3f" % s.stderr)+')' , font=('Helvetica', 15))
        s_label.grid(row = 3, column = 0)
        balance = s[0]*8000
        fab_label = Label(root, text = 'mean eaten ' + str('%.2f' % np.mean(nrgs))+
                      ' \n balance '+ str("%.3f" % balance), font=('Helvetica', 15))
        fab_label.grid(row = 4, column = 0)
    nrgs = look_for('nrgrecord.txt')
    nrgs = [i[1] for i in nrgs]
   
# Funzione per la seconda scheda
def scheda_2():
    # Creazione della lista
    # ...
    for widg in root.winfo_children():
        widg.grid_forget()
    # Creazione del bottone
    food_lable = Label(root, text='name of food ', font=('Helvetica', 15))
    food_lable.grid(row = 0, column = 0)

    food_entry = Entry(root, font=('Helvetica', 14))
    food_entry.grid(row=1, column=0)

    listed = Listbox(root, width = 40)
    listed.grid(row=2, column=0)

    def check(e):
        typed = the_entry.get()
        options.delete(0,END)
        if typed == '':
            for i in the_list:
                options.insert(END,i)
        else:
            for i in the_list:
                if typed.lower() in i.lower(): options.insert(END,i)
        return

    def fillout(e):
        the_entry.delete(0,END)
        the_entry.insert(0,options.get(ACTIVE))
        return

    food_entry.bind('<KeyRelease>', check)
    listed.bind('<<ListboxSelect>>', fillout)

    q_label = Label(root, text = 'eaten quantity', font = ('Helvetica, 15'))
    q_label.grid(row=0, column=1)

    q_entry = Entry(root, font=('Helvetica', 14))
    q_entry.grid(row=1, column=1)
    button_2 = Button(root, text="Torna alla scheda 1", command=scheda_1)
    button_2.grid(row=3, column=3)
    
    input_list = Listbox(root, width = 30)
    input_list.grid(row=2, column=2)

    def partial_cal(e):
        f = food_entry.get()
        q = q_entry.get()
        if q == '': q=0
        else: q=eval(q)
        if f in nvalues:
            if f in tte:
                cal = nvalues[f]['cal/hg']*q
            elif f in portions:
                cal = nvalues[f]['cal']*q
            message = 'adding '+str(cal)+' to calories'
            if root.grid_slaves(row=2, column=1)==[]:
                Label(root, text=message, font=('Helvetica', 15)).grid(row=2,column=1)
            else:
                root.grid_slaves(row=2, column=1)[0].grid_forget()
                Label(root, text=message, font=('Helvetica', 15)).grid(row=2,column=1)
            
        return

    q_entry.bind('<KeyRelease>', partial_cal)
    
    def add_to_eaten():
        #basically this takes a list of food and quantities and add to if the food and quantity you input
        global calories

        food = food_entry.get()
        quantity = q_entry.get()
        eaten_list.append([food,eval(quantity)])
        to_insert = quantity+' of '+food
        calories = comp_cal_eaten(eaten_list)
        input_list.delete(0,END)
        input_list.insert(END,'you ate today')
        for i in eaten_list:
            input_list.insert(END,str(i[1])+' of '+str(i[0]))
        input_list.insert(END,'so in total were eaten '+str(calories)+' kcal')
        return


    tte=look_for('tingstoeat.txt')
    portions=look_for('portions.txt')
    nvalues = look_for('nvalues.txt', dict())
    the_list=tte+portions
    the_entry=food_entry
    options=listed
    for i in the_list:
        options.insert(END,i)

    input_list.insert(END,'you ate today')
    for i in eaten_list:
        input_list.insert(END,str(i[1])+' of '+str(i[0]))
    if eaten_list==[]:
        input_list.insert(END,'nothing :(')
    else:
        input_list.insert(END,'so in total were eaten '+str(calories)+' kcal')

    the_button = Button(root, text='add', command = add_to_eaten)
    the_button.grid(row=1, column=2)
# Creazione della finestra principale
root = Tk()

def date_to_tuple(date):
    t = (date.year,date.month,date.day,date.hour,date.minute,date.second)
    return t

def tuple_to_date(t):
    date = d(t[0],t[1],t[2],t[3],t[4],t[5])
    return date

def look_for(path,default=[]):
    try:
        with open(path) as f:
            obj = f.read()
            f.close()
    except:
        print('nothing found at',path)
        obj=default
        return default
    return eval(obj)

def store_data(path,data):
    with open(path, 'w') as g:
        g.write(str(data))
        g.close()
    return

def comp_cal_eaten(eaten):
    tte=look_for('tingstoeat.txt')
    portions=look_for('portions.txt')
    nvalues = look_for('nvalues.txt', dict())
    calories = 0
    for i in eaten:
        if i[0] in tte:
            calories += nvalues[i[0]]['cal/hg']*i[1]
        elif i[0] in portions:
            calories += nvalues[i[0]]['cal']*i[1]
        else:
            top = Toplevel()
            top.title('do not know '+i[0])
            add_label = Label(top, text = 'to add to list', font = ('Helvetica', 12))
            add_label.grid(row=0,column=0)
            var=IntVar()
            Radiobutton(top, text='edible', variable=var, value=0).grid(row = 1, column = 0)
            Radiobutton(top, text='portion', variable=var, value=1).grid(row = 2, column = 0)
            cal_entry = Entry(top, font=('Helvetica', 20))
            cal_entry.grid(row=1, column=1)
            def get_cal():
                nonlocal calories
                cal_of_i = eval(cal_entry.get())
                calories += cal_of_i*i[1]
                top.destroy()
                return cal_of_i
            def the_add_func():
                nonlocal calories
                if var.get()==0:
                    tte.append(i[0])
                    tte.sort()
                    store_data('tingstoeat.txt',tte)
                    nvalues[i[0]] = {'cal/hg':eval(cal_entry.get())}
                    store_data('nvalues.txt', nvalues)
                    cal_of_i = get_cal()
                elif var.get()==1:
                    portions.append(i[0])
                    portions.sort()
                    store_data('portions.txt',portions)
                    nvalues[i[0]] = {'cal':eval(cal_entry.get())}
                    store_data('nvalues.txt', nvalues)
                    cal_of_i = get_cal()
                return
            add_button = Button(top, text='add to list', command = the_add_func)
            add_button.grid(row = 3, column = 0)
            once_label = Label(top, text = 'to just add calories', font = ('Helvetica', 12))
            once_label.grid(row=0,column=1)
            once_button = Button(top, text='add calories', command = get_cal)
            once_button.grid(row = 2, column = 1)

    return calories



date_today = d.now()


eaten_lists=look_for('eaten_lists.txt',[[(date_today.year,date_today.month,date_today.day),[]]])

if (date_today.year,date_today.month,date_today.day) == eaten_lists[-1][0]:
    eaten_list = eaten_lists[-1][1]
else:
    eaten_list = []
    eaten_lists.append([(date_today.year,date_today.month,date_today.day),[]])

calories = comp_cal_eaten(eaten_list)

# Impostazione del layout
root.grid_columnconfigure(0, weight=1)
root.grid_rowconfigure(0, weight=1)

# Visualizzazione della prima scheda
scheda_1()

# Avvio del mainloop
root.mainloop()
#olio 897 15/05

nothing found at tingstoeat.txt
nothing found at portions.txt
nothing found at wrecord.txt
nothing found at nrgrecord.txt
nothing found at tingstoeat.txt
nothing found at portions.txt
nothing found at wrecord.txt
nothing found at nrgrecord.txt


Cerchiamo di migliorare un po' la situazione utilizzando lambda e json.

In [18]:
import json
import os

def look_for(path,default=[]):
    if os.path.exists(path):
        with open("database.json", "r") as f:
            data = json.load(path)
    else:
        data = default
    return data

def record_weight(weight_value):
    weights = look_for('wrecord.json')
    weights.append([date_to_tuple(d.now()), weight_value])
    with open('wrecord.json', 'w') as f:
        json.dump(weights, f)
        
def record_nrg(energy):
    energy = look_for('nrgrecord.json')
    energy.append([date_to_tuple(d.now()), weight_value])
    with open('nrgrecord.json', 'w') as f:
        json.dump(energy, f)

def get_w():
    dates = []
    w = []
    wrec = look_for('wrecord.txt')
    for i in wrec:
        date=d(i[0][0],i[0][1],i[0][2],i[0][3]) -d(wrec[0][0][0],wrec[0][0][1],wrec[0][0][2],wrec[0][0][3])
        dates.append(date.total_seconds()/(3600*24))
        w.append(i[1])
    add_plot(np.array(dates),np.array(w))
    return np.array(dates),np.array(w)

def add_plot(x, y):
    fig = plt.Figure(figsize = (4, 4))
    splot = fig.add_subplot(111)
    splot.plot(y)
    canvas = FigureCanvasTkAgg(fig, master = root)  
    canvas.draw()
    canvas.get_tk_widget().grid(column=2)
    return

def clear_frame(self):
    for widget in self.main_container.winfo_children():
        widget.destroy()

def scheda_1():
    
    for widg in root.winfo_children():
        widg.grid_forget()
    
    w_button = Button(root, text = 'record weight today', command = lambda: record_weight(w_entry.get()))
    w_button.grid(row=2,column=0)

    nrg_lable = Label(root, text='eaten calories today', font=('Helvetica', 15))
    nrg_lable.grid(row = 0, column = 1)

    nrg_entry = Entry(root, font=('Helvetica', 20))
    nrg_entry.grid(row=1, column=1)
    #nrg_entry.insert(0,calories)
    
    w_lable = Label(root, text='weight today', font=('Helvetica', 15))
    w_lable.grid(row = 0, column = 0)

    w_entry = Entry(root, font=('Helvetica', 20))
    w_entry.grid(row=1, column=0)
    
    # Creazione del bottone
    button_1 = Button(root, text="Vai alla scheda 2", command=scheda_2)
    button_1.grid(row=3, column=3)
    
    nrg_button = Button(root, text = 'record calories today', command = lambda: record_nrg(nrg_entry.get()))
    nrg_button.grid(row=2,column=1)
    
    eaten_lists[-1][1] = eaten_list
    store_data('eaten_lists.txt',eaten_lists)
    
    gra = get_w()
    if 0<len(gra[0]):
        s = stats.linregress(gra[0],gra[1])
        s_label = Label(root, text=
                    str(" A = %.2f" % s[1])+'('+str("%.2f" % s.intercept_stderr)+') B = ' + 
                    str(" B = %.3f" % s[0])+'('+str("%.3f" % s.stderr)+')' , font=('Helvetica', 15))
        s_label.grid(row = 3, column = 0)
        balance = s[0]*8000
        fab_label = Label(root, text = 'mean eaten ' + str('%.2f' % np.mean(nrgs))+
                      ' \n balance '+ str("%.3f" % balance), font=('Helvetica', 15))
        fab_label.grid(row = 4, column = 0)
    nrgs = look_for('nrgrecord.txt')
    nrgs = [i[1] for i in nrgs]

In [19]:
root = Tk()
scheda_1()
root.mainloop()

In [20]:
p = (353)/(291+353)
print(p*260, p*30, 52*p, 110*p)

142.51552795031057 16.444099378881987 28.503105590062113 60.29503105590062


## The diet problem

Up to this point we did not talk at all about the diet problem which should be one of the main points of the project. i guess it is the moment to see something about it.

In [32]:
from pulp import*

tte = look_for('tingstoeat.txt')
portions = look_for('portions.txt')
nvalues = look_for('nvalues.txt', dict())
fcost = look_for('fcost.txt', dict())

calories = 0
proteins = 0
fat = 0
carb = 0

m = LpProblem('Diet_problem',LpMinimize)
variables = []

for i in tte:
    variables.append(LpVariable(i,0,None))
    if i in nvalues:
        if 'cal/hg' in nvalues[i]:
            calories += variables[-1]*nvalues[i]['cal/hg']
        if 'proteins/hg' in nvalues[i]:
            proteins += variables[-1]*nvalues[i]['proteins/hg']
        
for i in portions:
    variables.append(LpVariable(i,0,None,LpInteger))
    if i in nvalues:
        calories += variables[-1]*nvalues[i]['cal']

proteins

nothing found at fcost.txt


0.93*carote + 1.1*cipolla + 25*formaggio + 6.4*guanciale + 23*lenticchie + 1*maionese + 6.3*nutella + 9*pane + 0.9*passata_di_pomodoro + 12*pasta + 2*patate + 20*petto_di_pollo + 20*pollo + 22*salame_ungherese + 20*salmone + 2.3*sedano + 29*tonno + 13*uova + 11*uova_albume + 16*uova_tuorlo + 0.0

In [48]:
model = LpProblem("Balanced Diet Problem", LpMinimize)
x1 = LpVariable("wheat", 0, None, LpContinuous)
x2 = LpVariable("rice", 0, None, LpContinuous) 
x3 = LpVariable("corn", 0, None, LpContinuous)
model += 0.03 * x1 + 0.05 * x2 + 0.02 * x3 #adding an expression to the problem it becomes the objective
model += 4*x1 + 2*x2 + 2*x3 >= 27 #adding constraints they become constraints of the problem
model += 20*x1 + 25*x2 + 21*x3 >= 240 
model += 90*x1 + 110*x2 + 100*x3 >= 27 
model += x1 + x2 + x3 >= 12
model.solve()
print(model.objective, value(model.objective))
for i in model.variables(): print(i, value(i))

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/aa/.local/lib/python3.10/site-packages/pulp/solverdir/cbc/linux/64/cbc /tmp/f637ad8210f24e5799bbf2187b34b635-pulp.mps timeMode elapsed branch printingOptions all solution /tmp/f637ad8210f24e5799bbf2187b34b635-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 9 COLUMNS
At line 25 RHS
At line 30 BOUNDS
At line 31 ENDATA
Problem MODEL has 4 rows, 3 columns and 12 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 4 (0) rows, 3 (0) columns and 12 (0) elements
0  Obj 0 Primal inf 28.595454 (4)
2  Obj 0.255
Optimal - objective value 0.255
Optimal objective 0.255 - 2 iterations time 0.002
Option for printingOptions changed from normal to all
Total time (CPU seconds):       0.00   (Wallclock seconds):       0.00

0.02*corn + 0.05*rice + 0.03*wheat 0.255
corn 10.5
rice 0.0
wheat 1.5


## The diet in practice
The practical approach we want to adopt is having in the diet indications on what, how and when to eat. So we consider the possible preparations of the food.

In [3]:
prep=look_for('preparations.txt',dict())

nothing found at preparations.txt


In [4]:
prep['pasta al tonno']

{}